In [ ]:
pip install pandas numpy matplotlib scikit-learn pillow

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [58]:
INPUT_CSV = "D:/TRISHA/RESEARCH/tricycle_passenger_demand_data_collection_form.csv"
OUTPUT_DIR = r"D:\TRISHA\RESEARCH\Figures"
RANDOM_STATE = 42

In [59]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

Load and prepare data

In [60]:
df = pd.read_csv(INPUT_CSV)

In [61]:
time_order = [
    "7:00AM-8:00AM","8:00AM-9:00AM","9:00AM-10:00AM","10:00AM-11:00AM",
    "11:00AM-12:00PM","12:00PM-1:00PM","1:00PM-2:00PM","2:00PM-3:00PM",
    "3:00PM-4:00PM","4:00PM-5:00PM","5:00PM-6:00PM","6:00PM-7:00PM"
]
day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
weather_order = ["Sunny","Windy","Rainy","Stormy"]  # ascending severity
 
df["Time_of_Day_Code"] = df["Time_of_Day"].apply(lambda x: time_order.index(x) if x in time_order else -1)
df["Day_of_Week_Code"] = df["Day_of_Week"].apply(lambda x: day_order.index(x) if x in day_order else -1)
df["Weather_Code"] = df["Weather_Condition"].apply(lambda x: weather_order.index(x) if x in weather_order else -1)
 
feature_cols = [
    "Population",
    "Commercial_Activity_Score",
    "Distance_to_City_Center_km",
    "Time_of_Day_Code",
    "Day_of_Week_Code",
    "Weather_Code",
]
target_col = "Tricycle_Passenger_Count"
 
X = df[feature_cols].values
y = df[target_col].values
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

In [62]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

Train models

In [63]:
mlr = LinearRegression()
mlr.fit(X_train_s, y_train)
y_pred_mlr = mlr.predict(X_test_s)

In [64]:
ann = MLPRegressor(
    hidden_layer_sizes=(5,),
    activation="relu",
    solver="adam",
    max_iter=5000,
    random_state=RANDOM_STATE,
)
ann.fit(X_train_s, y_train)
y_pred_ann = ann.predict(X_test_s)

Evaluation metrics

In [65]:
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    nonzero = y_true != 0
    return np.mean(np.abs((y_true[nonzero] - y_pred[nonzero]) / y_true[nonzero])) * 100
 
metrics = {
    "MLR": {
        "R2": r2_score(y_test, y_pred_mlr),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_mlr)),
        "MAE": mean_absolute_error(y_test, y_pred_mlr),
        "MAPE": mape(y_test, y_pred_mlr),
    },
    "ANN": {
        "R2": r2_score(y_test, y_pred_ann),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_ann)),
        "MAE": mean_absolute_error(y_test, y_pred_ann),
        "MAPE": mape(y_test, y_pred_ann),
    },
}

In [66]:
print("Model performance:")
for model_name, m in metrics.items():
    print(f"  {model_name}: R2={m['R2']:.4f}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  MAPE={m['MAPE']:.2f}%")
 
plt.rcParams.update({"figure.dpi": 150, "font.size": 12})

Model performance:
  MLR: R2=0.8122  RMSE=5.57  MAE=4.38  MAPE=20.70%
  ANN: R2=0.8260  RMSE=5.36  MAE=4.23  MAPE=19.45%


Figure 1: Actual vs Predicted Demand (line, over test samples)

In [67]:
order = np.argsort(y_test)  
plt.figure(figsize=(9, 5))
plt.plot(np.array(y_test)[order], label="Actual Demand", color="black", linewidth=2)
plt.plot(np.array(y_pred_mlr)[order], label="MLR Predicted", color="tab:blue", linestyle="--")
plt.plot(np.array(y_pred_ann)[order], label="ANN Predicted", color="tab:red", linestyle=":")
plt.title("Figure 1. Actual vs. Predicted Tricycle Passenger Demand")
plt.xlabel("Test Sample (sorted by actual demand)")
plt.ylabel("Passenger Demand")
plt.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/figure1_actual_vs_predicted_demand.png")
plt.close()

Figure 2: MLR vs ANN R^2

In [68]:
plt.figure(figsize=(6, 5))
models = ["MLR", "ANN"]
r2_vals = [metrics[m]["R2"] for m in models]
bars = plt.bar(models, r2_vals, color=["tab:blue", "tab:red"])
plt.title("Figure 2. MLR vs. ANN — Coefficient of Determination (R²)")
plt.ylabel("R²")
plt.ylim(0, 1)
for b, v in zip(bars, r2_vals):
    plt.text(b.get_x() + b.get_width()/2, v + 0.02, f"{v:.3f}", ha="center")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/figure2_r2_comparison.png")
plt.close()

Figure 3: MLR vs ANN RMSE

In [69]:
plt.figure(figsize=(6, 5))
rmse_vals = [metrics[m]["RMSE"] for m in models]
bars = plt.bar(models, rmse_vals, color=["tab:blue", "tab:red"])
plt.title("Figure 3. MLR vs. ANN — Root Mean Square Error (RMSE)")
plt.ylabel("RMSE")
for b, v in zip(bars, rmse_vals):
    plt.text(b.get_x() + b.get_width()/2, v, f"{v:.2f}", ha="center", va="bottom")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/figure3_rmse_comparison.png")
plt.close()

Figure 4: MLR vs ANN MAE / MAPE (grouped bars, dual axis)

In [70]:
mae_vals = [metrics[m]["MAE"] for m in models]
mape_vals = [metrics[m]["MAPE"] for m in models]
colors = ["tab:blue", "tab:red"]

fig, (axL, axR) = plt.subplots(1, 2, figsize=(10, 5))

barsL = axL.bar(models, mae_vals, color=colors, width=0.5)
axL.set_title("Mean Absolute Error (MAE)")
axL.set_ylabel("MAE (passengers)")
axL.set_ylim(0, max(mae_vals) * 1.25)
for b, v in zip(barsL, mae_vals):
    axL.text(b.get_x() + b.get_width()/2, v, f"{v:.2f}", ha="center", va="bottom")

barsR = axR.bar(models, mape_vals, color=colors, width=0.5)
axR.set_title("Mean Absolute Percentage Error (MAPE)")
axR.set_ylabel("MAPE (%)")
axR.set_ylim(0, max(mape_vals) * 1.25)
for b, v in zip(barsR, mape_vals):
    axR.text(b.get_x() + b.get_width()/2, v, f"{v:.1f}%", ha="center", va="bottom")

plt.suptitle("Figure 4. MLR vs. ANN — MAE and MAPE")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/figure4_mae_mape_comparison.png")
plt.close()

Figure 5: Actual vs Predicted Scatter Plot

In [71]:
plt.figure(figsize=(6.5, 6.5))
plt.scatter(y_test, y_pred_mlr, alpha=0.6, label="MLR", color="tab:blue")
plt.scatter(y_test, y_pred_ann, alpha=0.6, label="ANN", color="tab:red")
lims = [min(y_test.min(), y_pred_mlr.min(), y_pred_ann.min()),
        max(y_test.max(), y_pred_mlr.max(), y_pred_ann.max())]
plt.plot(lims, lims, "k--", linewidth=1, label="Perfect Prediction")
plt.xlabel("Actual Demand")
plt.ylabel("Predicted Demand")
plt.title("Figure 5. Actual vs. Predicted Demand — Scatter Plot")
plt.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/figure5_actual_vs_predicted_scatter.png")
plt.close()

Figure 6: Passenger Demand Trend (across time and barangays)

In [72]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
 
hourly_avg = df.groupby("Time_of_Day")[target_col].mean().reindex(time_order)
axes[0].plot(range(len(hourly_avg)), hourly_avg.values, marker="o", color="tab:green")
axes[0].set_xticks(range(len(time_order)))
axes[0].set_xticklabels([t.split("-")[0] for t in time_order], rotation=45, ha="right")
axes[0].set_title("Average Demand by Hour (7AM–7PM)")
axes[0].set_ylabel("Avg. Passenger Demand")
 
barangay_avg = df.groupby("Clustered_Barangay")[target_col].mean().sort_values(ascending=False)
axes[1].bar(range(len(barangay_avg)), barangay_avg.values, color="tab:purple")
axes[1].set_xticks(range(len(barangay_avg)))
axes[1].set_xticklabels(barangay_avg.index, rotation=30, ha="right", fontsize=8)
axes[1].set_title("Average Demand by Clustered Barangay")
axes[1].set_ylabel("Avg. Passenger Demand")
 
plt.suptitle("Figure 6. Tricycle Passenger Demand Trend")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/figure6_demand_trend.png")
plt.close()

Figure 7: Residual / Error Plot

In [73]:
resid_mlr = y_test - y_pred_mlr
resid_ann = y_test - y_pred_ann
 
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(y_pred_mlr, resid_mlr, alpha=0.6, color="tab:blue")
axes[0].axhline(0, color="black", linestyle="--")
axes[0].set_title("MLR Residuals")
axes[0].set_xlabel("Predicted Demand")
axes[0].set_ylabel("Residual (Actual - Predicted)")
 
axes[1].scatter(y_pred_ann, resid_ann, alpha=0.6, color="tab:red")
axes[1].axhline(0, color="black", linestyle="--")
axes[1].set_title("ANN Residuals")
axes[1].set_xlabel("Predicted Demand")
axes[1].set_ylabel("Residual (Actual - Predicted)")
 
plt.suptitle("Figure 7. Residual / Error Plot")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/figure7_residual_plot.png")
plt.close()
 
print("\nAll 7 figures saved to", OUTPUT_DIR)


All 7 figures saved to D:\TRISHA\RESEARCH\Figures
